# AgriGuard: train the tomato-leaf classifier on Google Colab / Kaggle (free tier)

Runs the **same scripts** used locally (`ai_training/scripts`). Preprocessing is shared with the backend (`backend/app/ml/preprocessing.py`), so the exported model is drop-in compatible.

Dataset: PlantVillage tomato subset (CC BY-SA 3.0, Mohanty, Hughes & Salathé 2016). Runtime → *Change runtime type* → GPU speeds this up but CPU works (~20 min).

In [ ]:
# 1. Get the code (replace with your fork URL once the repo is on GitHub)
!git clone https://github.com/<your-account>/Agri-Guard.git
%cd Agri-Guard

In [ ]:
# 2. Download ONLY the three tomato classes + leaf map (sparse checkout, ~75 MB)
!git clone --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset.git ai_training/dataset/plantvillage_src
!cd ai_training/dataset/plantvillage_src && git sparse-checkout set --no-cone 'raw/color/Tomato___Early_blight/*' 'raw/color/Tomato___Late_blight/*' 'raw/color/Tomato___healthy/*' 'leaf-map.json'

In [ ]:
# 3. Dependencies (Colab already ships TensorFlow; this pins the rest)
!pip -q install scikit-learn matplotlib pillow

In [ ]:
# 4. Leakage-aware split (group by physical leaf + near-duplicates)
%cd ai_training/scripts
!python prepare_dataset.py

In [ ]:
# 5. Train (frozen head, then fine-tune top 30 layers)
!python train.py --epochs-head 8 --epochs-finetune 5

In [ ]:
# 6. Evaluate on the held-out test split -> reports/metrics.json + plots
!python evaluate.py

In [ ]:
# 7. Download the artefacts and copy them into ai_training/models/ of your local checkout
from google.colab import files
files.download('../models/agriguard_model.keras')
files.download('../models/agriguard_model_config.json')